# 04 SQL 商业分析

这一阶段用 DuckDB 在本地执行 SQL。

DuckDB 不需要额外打开一个独立软件。在这里，它的角色是一个本地数据库文件：`data/database/olist.duckdb`。

SQL 层的作用是把业务问题转成可复用、可审计的查询：

- KPI 口径
- 月度增长
- 物流体验
- 品类和地区表现
- 低评分风险分群


## 0. 导入依赖与路径


In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.options.display.float_format = "{:,.4f}".format

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
sql_dir = project_root / "sql"
database_path = project_root / "data" / "database" / "olist.duckdb"

database_path


## 1. 创建 / 刷新 DuckDB 数据库

这一步会运行 `sql/00_create_database.py`，把 processed CSV 导入 DuckDB，并创建几个常用 view。

In [ ]:
import runpy

runpy.run_path(sql_dir / "00_create_database.py", run_name="__main__")


## 2. 连接数据库


In [ ]:
con = duckdb.connect(database_path)

con.execute("SHOW TABLES").fetchdf()


## 3. 小工具：执行 SQL 文件


In [ ]:
def run_sql_file(filename: str) -> pd.DataFrame:
    sql_text = (sql_dir / filename).read_text(encoding="utf-8")
    return con.execute(sql_text).fetchdf()


## 4. Executive KPIs


In [ ]:
run_sql_file("01_business_kpis.sql")


## 5. 月度增长分析

这个查询展示窗口函数 `LAG()` 的使用，是数据分析面试里很常见的 SQL 能力点。

In [ ]:
growth = run_sql_file("02_growth_analysis.sql")
growth.head(10)


In [ ]:
growth.tail(10)


## 6. 物流体验与评价风险

这个查询把配送延迟分桶，然后观察每个分桶的低评分率。

In [ ]:
run_sql_file("03_delivery_experience.sql")


## 7. 品类和地区表现

`04_category_region_analysis.sql` 里有两段查询。DuckDB Python 默认一次返回最后一个结果集，所以这里拆开执行，更方便展示。

In [ ]:
category_region_sql = (sql_dir / "04_category_region_analysis.sql").read_text(encoding="utf-8")
category_sql, state_sql = [part.strip() for part in category_region_sql.split("-- Customer state performance.")]

category_result = con.execute(category_sql).fetchdf()
state_result = con.execute(state_sql).fetchdf()

category_result.head(20)


In [ ]:
state_result.head(20)


## 8. 低评分风险分群

这个查询把延迟和运费占比分成业务可解释的 segment，用来找高风险订单群体。

In [ ]:
run_sql_file("05_customer_review_risk.sql")


## 9. 自己写一条 SQL

下面这个单元可以用来修改和练习 SQL。示例问题：找出 GMV 最高的 10 个客户州 + 品类组合。

In [ ]:
custom_sql = """
SELECT
    customer_state,
    main_product_category,
    COUNT(*) AS orders,
    ROUND(SUM(payment_total), 2) AS gmv,
    ROUND(AVG(review_score_mean), 2) AS avg_review_score
FROM orders_analysis_base
WHERE customer_state IS NOT NULL
  AND main_product_category IS NOT NULL
GROUP BY customer_state, main_product_category
HAVING COUNT(*) >= 100
ORDER BY gmv DESC
LIMIT 10;
"""

con.execute(custom_sql).fetchdf()


## 10. 收尾

完成这一阶段后，项目已经具备：

- Python 数据清洗
- Python EDA 可视化
- DuckDB 本地数据库
- SQL 指标分析

下一步可以做 Dashboard，也可以先做机器学习模型。本项目先做低评分订单预测，因为它和前面的 EDA 结论衔接最自然。